# OBIS restructuring overview
This notebook restructures raw OBIS parquet files into a partitioned, storage-efficient layout (by layer/year/month) using DuckDB.

- Review `SOURCE_DIR` and `TARGET_DIR` before running. The script may delete and recreate the `TARGET_DIR`.
- The main steps: read schemas, union-by-name, normalize columns, spatial-sort, and export partitioned Parquet.

In [5]:
#!pip install duckdb
import duckdb
import os
import shutil

# Pfade definieren (Bitte anpassen!)
SOURCE_DIR = "/mnt/shared_data/finflow/obis_raw"  # Der Ordner, in dem die Überspezies-Ordner liegen
TARGET_DIR = "/mnt/shared_data/finflow/obis_structured"

# 1. ALTEN ORDNER KOMPLETT LÖSCHEN (falls existent)
if os.path.exists(TARGET_DIR):
    print(f"Lösche alten Ordner '{TARGET_DIR}' um Schema-Konflikte zu vermeiden...")
    shutil.rmtree(TARGET_DIR)

# 2. FRISCHEN ORDNER ERSTELLEN
os.makedirs(TARGET_DIR)
print(f"Frischer Zielordner '{TARGET_DIR}' wurde erstellt.")

# DuckDB Verbindung herstellen
con = duckdb.connect()
# RAM-Limit setzen, um Abstürze zusätzlich zu verhindern (Passe die GB an dein System an)
con.execute("PRAGMA memory_limit='16GB'")

print(f"Lese von: {SOURCE_DIR}/*/*/*.parquet")
print(f"Schreibe nach: {TARGET_DIR}")

Lösche alten Ordner '/mnt/shared_data/finflow/obis_structured' um Schema-Konflikte zu vermeiden...
Frischer Zielordner '/mnt/shared_data/finflow/obis_structured' wurde erstellt.
Lese von: /mnt/shared_data/finflow/obis_raw/*/*/*.parquet
Schreibe nach: /mnt/shared_data/finflow/obis_structured


In [9]:
import duckdb

# Pfad zu den Rohdaten anpassen!
con = duckdb.connect()

print("Lese Parquet-Metadaten (Footers) ein... das dauert nur kurz und schont den RAM.")

# Query, die nur das Schema (die Struktur) lädt, ohne Zeilen zu extrahieren
query = f"""
SELECT * FROM read_parquet('{TARGET_DIR}/*/*/*/*.parquet', union_by_name=true)
LIMIT 0;
"""

# Führe die Query aus
result = con.execute(query)

# Die Spaltennamen stecken in der 'description' des Ergebnisses
columns = [desc[0] for desc in result.description]

print(f"\nErfolg! Es wurden insgesamt {len(columns)} verschiedene Spalten gefunden.\n")
print("Hier ist die alphabetische Liste aller verfügbaren Spalten:")
print("-" * 50)

# Alphabetisch sortiert ausgeben, damit du sie in Ruhe durchgehen kannst
for col in sorted(columns):
    print(f"- {col}")

Lese Parquet-Metadaten (Footers) ein... das dauert nur kurz und schont den RAM.

Erfolg! Es wurden insgesamt 14 verschiedene Spalten gefunden.

Hier ist die alphabetische Liste aller verfügbaren Spalten:
--------------------------------------------------
- basisOfRecord
- decimalLatitude
- decimalLongitude
- depth
- individualCount
- layer
- month
- occurrenceID
- shoredistance
- species
- sss
- sst
- vernacularName
- year


In [6]:
# Der SQL-Befehl für die Umstrukturierung - RAM-optimiert auf die FinFlow-Spalten
query = f"""
COPY (
    WITH raw_data AS (
        SELECT 
            -- 1. Pflichtspalten für die Karte
            decimalLongitude,
            decimalLatitude,
            eventDate,
            species,
            
            -- 2. Detail-Spalten für UI/Popups (werden mit NULL gefüllt, falls in einer Datei nicht vorhanden)
            vernacularName,
            occurrenceID,
            individualCount,
            basisOfRecord,
            
            -- 3. Umwelt-Kontext (Optional, aber cool)
            depth,
            sst,
            sss,
            shoredistance,
            
            -- Layer extrahieren
            list_extract(string_split(filename, '/'), -3) AS layer, 
            
            -- Zeitstempel aufbereiten
            EXTRACT(year FROM CAST(eventDate AS TIMESTAMP)) AS year,
            LPAD(CAST(EXTRACT(month FROM CAST(eventDate AS TIMESTAMP)) AS VARCHAR), 2, '0') AS month
            
        -- union_by_name=true rettet uns hier, da wir spezifische Spalten fordern, 
        -- die vielleicht nicht in jedem einzelnen Chunk existieren.
        FROM read_parquet('{SOURCE_DIR}/*/*/*.parquet', filename=true, union_by_name=true)
        
        -- Datenbereinigung
        WHERE 
            decimalLongitude BETWEEN -180 AND 180
            AND decimalLatitude BETWEEN -90 AND 90
            AND eventDate IS NOT NULL
    )
    SELECT * EXCLUDE (eventDate) -- Das rohe eventDate werfen wir am Ende weg, da wir year/month partitionieren
    FROM raw_data
    
    -- SPATIAL SORTING (Grid-Sortierung für blitzschnelles Laden der Karte)
    ORDER BY 
        layer, 
        year, 
        month, 
        CAST(FLOOR(decimalLongitude * 10) AS INT), 
        CAST(FLOOR(decimalLatitude * 10) AS INT),
        decimalLongitude, 
        decimalLatitude 
        
) TO '{TARGET_DIR}' (
    FORMAT PARQUET,
    PARTITION_BY (layer, year, month),
    OVERWRITE_OR_IGNORE 1
);
"""

print("Starte speicheroptimierte Restrukturierung...")
con.execute(query)
print("Restrukturierung erfolgreich!")

Starte extrem speicheroptimierte Restrukturierung...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Restrukturierung erfolgreich abgeschlossen! Eure Parquet-Dateien sind jetzt winzig und rasend schnell.


In [7]:
# 1. Zeige die erstellten Layer-Ordner an
print("Erstellte Layer:")
print(os.listdir(TARGET_DIR))

# 2. Mache eine schnelle Test-Abfrage auf die NEUEN Daten
test_query = f"""
SELECT layer, year, month, count(*) as anzahl_sichtungen
FROM read_parquet('{TARGET_DIR}/*/*/*/*.parquet')
GROUP BY layer, year, month
ORDER BY anzahl_sichtungen DESC
LIMIT 5;
"""

print("\nTop 5 Monate mit den meisten Daten:")
display(con.execute(test_query).df())

Erstellte Layer:
['layer=Sphenisciformes', 'layer=Gadidae', 'layer=Xiphiidae', 'layer=Cetacea', 'layer=Otariidae', 'layer=Cheloniidae', 'layer=Holocephali', 'layer=Sirenia', 'layer=Odobenidae', 'layer=Dermochelyidae', 'layer=Istiophoridae', 'layer=Elasmobranchii', 'layer=Cephalopoda', 'layer=Scombridae', 'layer=Phocidae', 'layer=Clupeidae', 'layer=Pleuronectiformes']

Top 5 Monate mit den meisten Daten:


,layer,year,month,anzahl_sichtungen
0,Sphenisciformes,2011,10,65141
1,Clupeidae,2003,03,45023
2,Sphenisciformes,2012,12,42409
3,Cetacea,2007,09,39487
4,Clupeidae,2002,03,39283
